# Notebook 09: Which genes carry the decoupling? (gene-level rewiring)

**Goal.** Find the mitophagy and lysosome genes whose link to the other arm weakens most with age: the **interface genes**. These become your candidate list.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** For every mouse, computes the adjusted correlation between *each* mitophagy gene and *each* lysosome gene (19 x 44 pairs; Fisher-z scale). The result is one small matrix per mouse. Nothing is averaged over cells that belong to a different mouse.

*You should see:* 'SIM_A 11 samples' and 'SIM_B 8 samples'.

In [ ]:
from mlc.dataset import Dataset
from mlc.analysis import cross_arm_z, rewiring_test
from mlc import plots
sheet = pd.read_csv(SHEET).set_index("sample_id")
Z, cohs = {}, {}
for c in cohorts()["mouse"]:
    ds = Dataset.load(PROC / f"{c}_scored")
    cv = ["depth", "pct_mt", "dissoc"] + [x for x in ds.obs.columns if x.startswith("state_")]
    z = cross_arm_z(ds, G.MITOPHAGY, G.LYSOSOME, cv, min_cells=MIN_CELLS)
    Z.update(z); cohs.update({s: c for s in z}); print(c, len(z), "samples")
ages = [float(sheet.loc[s, "age_months"]) for s in Z]; coh = [cohs[s] for s in Z]

**Cell 3.** **delta** for a gene = its average link to the other arm in young mice minus in old mice (positive = weaker with age). Young/old are defined within each cohort (below or above the cohort's median age). The p-value comes from shuffling the young/old labels 1,000 times *within cohort* (so the replicated design is respected), then Benjamini-Hochberg **fdr** controls false discoveries across all genes. Use n_perm=5000 or more for real analyses.

*You should see:* Top genes include Ctsd, Lamp2, Mcoln1, Atp6v0a1, Lamp1 (lysosome) and Pink1, Bnip3l (mitophagy); FDR about 0.02-0.03.

In [ ]:
tbl, Mj, Mo, A, B = rewiring_test(Z, ages, coh, n_perm=1000, seed=SEED)
tbl.to_csv(RES / "rewiring_genes.csv", index=False)
print(tbl.head(12).round(4))

**Cell 4.** **Figure 4a-c.** Heat maps of the average cross-arm link in young and old mice (rows mitophagy genes, columns lysosome genes; red = positive link): the bright block fades with age. Panel c ranks genes by loss of coupling. Genes with a small fdr are your candidate list for notebooks 13 to 15.

In [ ]:
plots.fig_rewire(Mj, Mo, A, B, tbl); plt.show()

**Cell 5.** **Simulated data only:** checks the method against the known answer. The top 14 genes should contain most of the 14 planted interface genes. On real data there is no such check, which is why the permutation p-values and replication matter.

*You should see:* 11 of 14 planted genes recovered (about 10-13 is typical).

In [ ]:
import json
truth = json.load(open(RAW / "sim" / "truth.json"))
true_set = set(truth["interface_mito"] + truth["interface_lyso"])
top = set(tbl.head(14).gene)
print("simulated interface genes found in the top 14:", len(top & true_set), "of", len(true_set), sorted(top & true_set))